# Medical Image Segmentation: Region Growing

This notebook demonstrates segmentation methods for extracting topologically connected structures in biomedical images (knee joint MRI scan). The implementation is based on the **Region Growing** algorithm, using local neighborhood analysis and similarity predicates.

The project covers:
- **Low-level DFS (Depth-First Search) implementation:** Stack-based iterative traversal of the image matrix (avoiding call-stack overflow from deep recursion).
- **Static thresholding segmentation:** Pixel classification based on a strict intensity deviation tolerance from the initial seed point.
- **Dynamic mean thresholding segmentation:** Algorithm refinement that compensates for natural intensity gradients (shading) in MRI scans via O(1) running mean updates of the region luminance.
- **Pre-processing:** Low-pass filtering (Gaussian blur) to limit segmentation leakage through thin noise bridges.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob

# Load input data (MRI scan)
if not os.path.exists("knee.png"):
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/12_Segmentation/knee.png --no-check-certificate

im = cv2.imread('knee.png', cv2.IMREAD_GRAYSCALE)

# Initialize seed point
x0, y0 = 100, 250

fig, ax = plt.subplots(figsize=(6, 6))
circle = plt.Circle((x0, y0), 5, color='r', fill=True)
ax.imshow(im, cmap='gray')
ax.add_patch(circle)
plt.title("Input MRI Scan - Seed Point")
plt.axis('off')
plt.show()

## 1. Fixed Similarity Predicate Algorithm

Classic approach: the algorithm examines the 8-connected Moore neighborhood of each pixel popped from the stack. If the intensity difference between a neighbor and the current pixel is below the defined threshold (T=4), the pixel is added to the segment. A `visited` matrix prevents cycles.

In [ ]:
def region_growing_static(image, seed_x, seed_y, threshold=4):
    """
    Isolates a structure using a strict inter-pixel deviation threshold.
    """
    h, w = image.shape
    vis = np.zeros((h, w), dtype=bool)
    seg = np.zeros((h, w), dtype=bool)
    
    st = [(seed_y, seed_x)]
    vis[seed_y, seed_x] = True
    seg[seed_y, seed_x] = True
    
    # Cast to int to avoid uint8 arithmetic overflow
    im_i = image.astype(int)
    
    while st:
        y, x = st.pop()
        
        # Guard against matrix boundaries
        if y < 1 or y >= h - 1 or x < 1 or x >= w - 1:
            continue
            
        for i in range(-1, 2):
            for j in range(-1, 2):
                ny, nx = y + i, x + j
                if not vis[ny, nx]:
                    # Evaluate static predicate Q
                    if abs(im_i[y, x] - im_i[ny, nx]) < threshold:
                        seg[ny, nx] = True
                        st.append((ny, nx))
                    vis[ny, nx] = True
                    
    return seg

seg_static = region_growing_static(im, x0, y0, threshold=4)

plt.figure(figsize=(6, 6))
plt.imshow(seg_static, cmap='gray')
plt.title("Segmentation (T=4) - Noticeable Structure Fragmentation")
plt.axis('off')
plt.show()

## 2. Adaptive Architecture (Dynamic Mean Thresholding)

A threshold that is too narrow yields incomplete segmentation, while increasing it without adaptation causes leakage into background structures due to intensity gradients in the image.

The approach below implements a running update of the mean over the entire extracted region. The new predicate measures pixel deviation not from its immediate neighbor, but from the updated mean, which gives the model high stability.

In [ ]:
def region_growing_dynamic(image, seed_x, seed_y, threshold=35):
    """
    Improved algorithm that adapts to noise and contrast variation in bone tissue.
    """
    # Preprocessing: noise reduction to prevent segmentation leakage
    blur = cv2.GaussianBlur(image, (5, 5), 0).astype(int)
    
    h, w = blur.shape
    vis = np.zeros((h, w), dtype=bool)
    seg = np.zeros((h, w), dtype=bool)
    
    st = [(seed_y, seed_x)]
    vis[seed_y, seed_x] = True
    seg[seed_y, seed_x] = True
    
    # Parameters for tracking the dynamic mean
    mv = blur[seed_y, seed_x]
    ns = 1
    
    while st:
        y, x = st.pop()
        
        if y < 1 or y >= h - 1 or x < 1 or x >= w - 1:
            continue
            
        for i in range(-1, 2):
            for j in range(-1, 2):
                ny, nx = y + i, x + j
                if not vis[ny, nx]:
                    # New predicate Q based on deviation from the region mean
                    if abs(blur[ny, nx] - mv) < threshold:
                        seg[ny, nx] = True
                        st.append((ny, nx))
                        
                        # O(1) incremental mean update
                        ns += 1
                        mv = (mv * (ns - 1) + blur[ny, nx]) / ns
                        
                    vis[ny, nx] = True
                    
    return seg

seg_dynamic = region_growing_dynamic(im, x0, y0, threshold=35)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(seg_static, cmap='gray')
ax[0].set_title("Naive Approach: Fixed Neighborhood (Threshold T=4)")
ax[0].axis('off')

ax[1].imshow(seg_dynamic, cmap='gray')
ax[1].set_title("Adaptive Approach: Blur + Mean-Based Predicate (Threshold T=35)")
ax[1].axis('off')
plt.show()

### Workspace Cleanup

In [ ]:
trash_files = glob.glob('knee.png')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Temporary test images removed from the filesystem.")